# 🌊 Rastreador de Cachoeiras
Digite uma cidade e o notebook procura, por satélite (Google Earth Engine), locais com **probabilidade de ter cachoeira**.
Cada busca fica salva no seu Google Drive.

**Como usar (no celular também):** execute as células em ordem tocando em ▶. Na célula 3 preencha a cidade e execute.

## 1. Instalar

In [ ]:
!pip -q install folium anthropic

## 2. Configuração
- **PROJETO_EE**: ID do seu projeto Google Cloud registrado no Earth Engine (https://code.earthengine.google.com/register, uso não comercial é grátis).
- **Conferência por IA (opcional)**: guarde sua chave em *Secrets* (ícone 🔑 na lateral esquerda) com o nome `ANTHROPIC_API_KEY` e ative o acesso ao notebook.

In [ ]:
#@title Configuração { display-mode: "form" }
PROJETO_EE = ""  #@param {type:"string"}
SALVAR_NO_DRIVE = True  #@param {type:"boolean"}
USAR_IA = True  #@param {type:"boolean"}

## 3. Entrar no Earth Engine e no Drive

In [ ]:
import ee, os, json, math, time, base64, requests

assert PROJETO_EE, "Preencha PROJETO_EE na célula 2 e execute-a de novo."
ee.Authenticate()
ee.Initialize(project=PROJETO_EE)

ARQ = "/content/buscas.json"
if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/cachoeiras", exist_ok=True)
    ARQ = "/content/drive/MyDrive/cachoeiras/buscas.json"

CHAVE_IA = None
if USAR_IA:
    try:
        from google.colab import userdata
        CHAVE_IA = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        print("Sem chave da IA em Secrets: seguindo só com a nota por relevo e rios.")
print("Pronto. Buscas salvas em:", ARQ)

## 4. Funções (detector, probabilidade, IA)

In [ ]:
UA = {"User-Agent": "rastreador-cachoeiras/0.1"}
MAX_GRAUS = 0.4

def geocodificar(cidade):
    r = requests.get("https://nominatim.openstreetmap.org/search", headers=UA, timeout=20,
                     params={"q": cidade, "format": "json", "limit": 1, "countrycodes": "br"})
    r.raise_for_status()
    d = r.json()
    if not d:
        return None
    sul, norte, oeste, leste = map(float, d[0]["boundingbox"])
    lat, lng = float(d[0]["lat"]), float(d[0]["lon"])
    meio = min(max(norte - sul, leste - oeste, 0.15) / 2, MAX_GRAUS / 2)
    return {"nome": d[0]["display_name"], "lat": lat, "lng": lng,
            "bbox": [lng - meio, lat - meio, lng + meio, lat + meio]}

def pontuar(p):
    f_queda = min(p["queda_m"] / 80.0, 1.0)
    f_agua = min(max(math.log10(max(p["area_km2"], 1.0) / 10.0), 0.0) / 2.0, 1.0)
    return round(100 * (0.6 * f_queda + 0.4 * f_agua))

def deduplicar(pontos, grau=0.003):
    melhor = {}
    for p in pontos:
        k = (round(p["lat"] / grau), round(p["lng"] / grau))
        if k not in melhor or p["queda_m"] > melhor[k]["queda_m"]:
            melhor[k] = p
    return list(melhor.values())

def detectar(bbox, min_area=10.0, queda=25.0, limite=3000):
    regiao = ee.Geometry.Rectangle(bbox)
    elev = ee.Image("USGS/SRTMGL1_003").select("elevation")
    upa = ee.Image("MERIT/Hydro/v1_0_1").select("upa")
    rio = upa.gte(min_area)
    desnivel = elev.subtract(elev.focalMin(radius=150, units="meters")).rename("queda_m")
    img = ee.Image.cat([desnivel, upa.rename("area_km2")]).updateMask(rio.And(desnivel.gte(queda)))
    feats = img.sample(region=regiao, scale=30, geometries=True, numPixels=limite).getInfo()["features"]
    pts = []
    for f in feats:
        lng, lat = f["geometry"]["coordinates"]
        pr = f["properties"]
        pts.append({"lat": round(lat, 5), "lng": round(lng, 5),
                    "queda_m": round(pr["queda_m"]), "area_km2": round(pr["area_km2"], 1)})
    pts = deduplicar(pts)
    for p in pts:
        p["probabilidade"] = pontuar(p)
    return sorted(pts, key=lambda p: -p["probabilidade"])

def imagem_satelite(lat, lng, d=0.004):
    r = requests.get(
        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/export",
        headers=UA, timeout=30,
        params={"bbox": f"{lng-d},{lat-d},{lng+d},{lat+d}", "bboxSR": 4326, "imageSR": 4326,
                "size": "512,512", "format": "jpg", "f": "image"})
    r.raise_for_status()
    return r.content

def conferir_com_ia(pontos, limite=10):
    if not CHAVE_IA:
        return False
    import anthropic
    cli = anthropic.Anthropic(api_key=CHAVE_IA)
    for p in pontos[:limite]:
        try:
            img = base64.b64encode(imagem_satelite(p["lat"], p["lng"])).decode()
            msg = cli.messages.create(
                model="claude-sonnet-5-5", max_tokens=300,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img}},
                    {"type": "text", "text":
                        "Imagem de satélite (~800 m) centrada num trecho de rio com forte desnível. "
                        "Estime a chance de haver uma cachoeira natural (não barragem, vertedouro ou "
                        "corredeira suave). Responda só JSON: "
                        '{"probabilidade": 0-100, "motivo": "uma frase em português"}'}]}])
            t = msg.content[0].text
            ia = json.loads(t[t.index("{"): t.rindex("}") + 1])
            p["ia_probabilidade"] = int(ia["probabilidade"])
            p["ia_motivo"] = ia["motivo"]
            p["probabilidade"] = round((p["probabilidade"] + p["ia_probabilidade"]) / 2)
        except Exception as e:
            p["ia_motivo"] = f"IA indisponível: {type(e).__name__}"
    pontos.sort(key=lambda p: -p["probabilidade"])
    return True

def carregar_buscas():
    try:
        return json.load(open(ARQ, encoding="utf-8"))
    except FileNotFoundError:
        return []

def salvar_busca(consulta, resultado):
    buscas = carregar_buscas()
    resultado["id"] = len(buscas) + 1
    resultado["consulta"] = consulta
    resultado["criado_em"] = time.time()
    buscas.append(resultado)
    json.dump(buscas, open(ARQ, "w", encoding="utf-8"), ensure_ascii=False)
    return resultado["id"]

def desenhar_mapa(res):
    import folium
    loc = res["local"]
    m = folium.Map(location=[loc["lat"], loc["lng"]], zoom_start=11, tiles=None)
    folium.TileLayer(
        "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery", name="Satélite").add_to(m)
    for i, p in enumerate(res["candidatas"], 1):
        cor = "red" if p["probabilidade"] >= 70 else "orange" if p["probabilidade"] >= 50 else "yellow"
        txt = (f"<b>#{i} · {p['probabilidade']}%</b><br>Queda ~{p['queda_m']} m · "
               f"drenagem {p['area_km2']} km²" + (f"<br><i>IA: {p['ia_motivo']}</i>" if p.get("ia_motivo") else ""))
        folium.CircleMarker([p["lat"], p["lng"]], radius=7, color=cor, fill=True, fill_opacity=.85,
                            popup=folium.Popup(txt, max_width=260)).add_to(m)
    b = loc["bbox"]
    m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
    return m

## 5. Procurar cachoeiras
Digite a cidade e execute. Leva de 30 s a 1 min.

In [ ]:
#@title Buscar { display-mode: "form" }
CIDADE = "Bento Gonçalves, RS"  #@param {type:"string"}

from IPython.display import display
local = geocodificar(CIDADE)
if not local:
    raise SystemExit(f"Cidade '{CIDADE}' não encontrada.")
print("Analisando", local["nome"].split(",")[0], "…")
pontos = detectar(local["bbox"])[:50]
ia = conferir_com_ia(pontos)
res = {"local": local, "ia": ia, "candidatas": pontos}
salvar_busca(CIDADE, res)
print(f"{len(pontos)} locais prováveis" + (" (conferidos por IA)" if ia else " (sem conferência por IA)") + " · busca salva.")
for i, p in enumerate(pontos[:10], 1):
    print(f"#{i}  {p['probabilidade']}%  queda ~{p['queda_m']} m  ({p['lat']}, {p['lng']})  {p.get('ia_motivo','')}")
display(desenhar_mapa(res))

## 6. Buscas salvas
Lista as buscas anteriores; coloque o número e execute para reabrir sem refazer a análise.

In [ ]:
#@title Reabrir busca salva { display-mode: "form" }
NUMERO = 0  #@param {type:"integer"}
import time
from IPython.display import display
buscas = carregar_buscas()
if not buscas:
    print("Nenhuma busca salva ainda.")
for b in buscas:
    print(f"{b['id']:>3} · {b['consulta']} · {len(b['candidatas'])} locais · "
          + time.strftime("%d/%m/%Y %H:%M", time.localtime(b["criado_em"])))
if NUMERO:
    escolhida = next((b for b in buscas if b["id"] == NUMERO), None)
    print("Busca não encontrada.") if not escolhida else display(desenhar_mapa(escolhida))